# 03b. ModernBERT on Kaggle

Training run for the ModernBERT section of `03a_bert_finetune.ipynb`, executed on Kaggle with two T4 GPUs because the laptop GPU (6 GB) and RAM were not enough. It installs the `bankguard` package from GitHub, builds the same features and sentences (the MD5 hashes match the laptop), fine-tunes ModernBERT-base, and saves the validation scores and metrics that `03a_bert_finetune.ipynb` analyses.

This is the second run, with 4 epochs and evaluation every 4,000 steps. The first run (2 epochs, evaluation every 2,000 steps) is the previous version of this file in the git history and Version 1 on Kaggle.

In [1]:
import sys

import torch
import transformers

print("python:", sys.version.split()[0])
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(), "| gpus:", torch.cuda.device_count())
print("gpu:", torch.cuda.get_device_name(0))
print("transformers:", transformers.__version__)
!free -g | head -2

python: 3.13.16
torch: 2.11.0+cu130 | cuda: True | gpus: 2
gpu: Tesla T4
transformers: 5.20.2
               total        used        free      shared  buff/cache   available
Mem:              31           1          24           0           5          29


In [2]:
!pip install --no-deps --force-reinstall git+https://github.com/sap6011/bank-guard.git

  Cloning https://github.com/sap6011/bank-guard.git to /tmp/pip-req-build-hn4ks3te
  Running command git clone --filter=blob:none --quiet https://github.com/sap6011/bank-guard.git /tmp/pip-req-build-hn4ks3te
  Resolved https://github.com/sap6011/bank-guard.git to commit e5a22820d1cea9bf913d25fc6baf702065e9a635
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for bank-guard: filename=bank_guard-0.1.0-py3-none-any.whl size=9575 sha256=972816393db91f39e2c93cb94a91adc0a1532a10849d280da9fbd853bb5d0f68
  Stored in directory: /tmp/pip-ephem-wheel-cache-5smrrc6_/wheels/64/0a/d6/432fce36897390a68ec24ef5723db09af1dba911df99f3f0f1
Successfully built bank-guard


In [3]:
from bankguard.features import FEATURES
from bankguard.text import to_text

print("ok", len(FEATURES))

ok 22


In [4]:
import hashlib
from pathlib import Path

import numpy as np
import torch

from bankguard.data import load_accounts, load_transactions, split_by_date
from bankguard.features import FEATURES, build_features, fit_currency_median

DATA = Path("/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml")

tx = load_transactions(DATA / "HI-Small_Trans.csv")
accounts = load_accounts(DATA / "HI-Small_accounts.csv")
splits = split_by_date(tx)
train, val = splits["train"], splits["validation"]

currency_median = fit_currency_median(train)
X = build_features(tx, accounts, currency_median)
X_train, y_train = X.loc[train.index, FEATURES], train["is_laundering"].to_numpy()
X_val, y_val = X.loc[val.index, FEATURES], val["is_laundering"].to_numpy()
print(X_train.shape, y_train.sum(), X_val.shape, y_val.sum())

rng = np.random.default_rng(42)
pos = np.flatnonzero(y_train == 1)
neg = rng.choice(np.flatnonzero(y_train == 0), size=200_000, replace=False)
sample = np.sort(np.concatenate([pos, neg]))

train_text = to_text(X_train.iloc[sample]).tolist()
train_labels = y_train[sample]
val_text = to_text(X_val).tolist()

print("train text:", hashlib.md5("\n".join(train_text).encode()).hexdigest())
print("val text:  ", hashlib.md5("\n".join(val_text).encode()).hexdigest())

(3248921, 22) 2530 (965524, 22) 1036
train text: 6b7502e690497d6d06dbd6ae12a15e96
val text:   ed4defe73ab6fe9ee28e95ea9e7f12b3


In [5]:
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

MODEL = "answerdotai/ModernBERT-base"
MAX_LEN = 128
tokenizer = AutoTokenizer.from_pretrained(MODEL)


class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts, self.labels = texts, labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        return {"text": self.texts[i], "label": int(self.labels[i])}


def collate(batch):
    enc = tokenizer(
        [b["text"] for b in batch],
        truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt",
        return_token_type_ids=False,
    )
    enc["labels"] = torch.tensor([b["label"] for b in batch])
    return enc


train_ds = TextDataset(train_text, train_labels)

config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

In [6]:
import time

from sklearn.metrics import average_precision_score

# The same evaluation fifth as on the laptop (same seed).
eval_idx = np.random.default_rng(0).choice(len(val_text), size=200_000, replace=False)
eval_ds = TextDataset([val_text[i] for i in eval_idx], y_val[eval_idx])


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    scores = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    return {"pr_auc": average_precision_score(labels, scores)}


model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)

args = TrainingArguments(
    output_dir="/kaggle/temp/modernbert",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=500,
    fp16=True,
    eval_strategy="steps",
    eval_steps=4000,
    save_strategy="steps",
    save_steps=4000,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="pr_auc",
    greater_is_better=True,
    logging_steps=200,
    remove_unused_columns=False,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds,
    data_collator=collate, compute_metrics=compute_metrics,
)

start = time.time()
result = trainer.train()
train_minutes = (time.time() - start) / 60
print(f"training time: {train_minutes:.1f} min")

model.safetensors: reconstructing file:   0%|          |  0.00B /  599MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss,Pr Auc
4000,0.082512,0.015077,0.113021
8000,0.080190,0.011256,0.231405
12000,0.073891,0.016476,0.277802
16000,0.057012,0.012243,0.283101
20000,0.048447,0.011594,0.316349
24000,0.070227,0.013481,0.330445
25320,0.060671,0.012905,0.331816


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (9 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

training time: 386.9 min


In [7]:
import json

from bankguard.evaluate import evaluate

pred = trainer.predict(TextDataset(val_text, y_val))
val_scores = torch.softmax(torch.tensor(pred.predictions), dim=-1)[:, 1].numpy()
np.save("/kaggle/working/modernbert_4ep_val_scores.npy", val_scores)

trainer.save_model("/kaggle/working/modernbert_4ep_best")
tokenizer.save_pretrained("/kaggle/working/modernbert_4ep_best")

metrics = evaluate(y_val, val_scores, "ModernBERT, 4 epochs")
metrics["train_minutes"] = train_minutes
metrics["best_checkpoint"] = trainer.state.best_model_checkpoint
metrics["log_history"] = trainer.state.log_history
with open("/kaggle/working/modernbert_4ep_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print({k: v for k, v in metrics.items() if k != "log_history"})

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'model': 'ModernBERT, 4 epochs', 'pr_auc': 0.3525596803772848, 'recall_at_0.1%_fpr': 0.3754826254826255, 'recall_at_1.0%_fpr': 0.7606177606177607, 'train_minutes': 386.9274162610372, 'best_checkpoint': '/kaggle/temp/modernbert/checkpoint-25320'}
